# Day 07：第 1 周复盘与综合实战 —— 企业通讯录 / Agent 联系人管理系统

> **🎙️ 讲师旁白**：欢迎来到星辰数科《智枢企业级智能体平台》第 7 天！今天是第 1 周的收官 Capstone 实战。我们将把本周学到的变量与数字名片（Day 01）、字符串清洗（Day 02）、条件与循环（Day 03）、列表与字典（Day 04）、JSON 序列化（Day 05）以及 `pathlib` 异常防御（Day 06）全部融会贯通，亲手跑通 **企业通讯录 / Agent 联系人管理系统** 的完整 CRUD + JSON 持久化闭环！

## 1. 第 1 周核心能力自检：导入智枢平台 Day 07 模块
首先把 `project_iterations/day_07` 加入模块搜索路径，并验证继承自 Day 01 的环境自检与数字名片。

In [1]:
import sys
from pathlib import Path

repo_root = Path.cwd()
for parent in [repo_root, *repo_root.parents]:
    candidate = parent / 'project_iterations' / 'day_07'
    if candidate.exists():
        sys.path.insert(0, str(candidate))
        break

from zhishu_agent.profile import get_environment_info, build_employee_card
from zhishu_agent.contacts import (
    add_contact, find_contact, update_contact, delete_contact,
    list_contacts, save_contacts, load_contacts, format_contacts_table,
)

env = get_environment_info()
print('Python 版本:', env['python_version'])
print(build_employee_card('陈默', 'AI 应用开发工程师', 2, 18.5, True))

Python 版本: 3.11.2
   星辰数科 · 智枢平台数字名片 (v0.1.0-day07)
姓名 / 标识 : 陈默
所属岗位    : AI 应用开发工程师
工程经验    : 2 年
资源预算    : ¥18.50 / 小时
大模型权限  : 已开通 [OK]


## 2. Create（新增）：录入真人研发成员与 AI Agent 节点
我们初始化一个空通讯录列表 `contacts`，通过 `add_contact()` 录入员工与 AI Agent。函数会自动清洗首尾空格、校验 `agent_type` 合法性并防止重名。

In [2]:
contacts = []

add_contact(contacts, name='张工', role='Tech Lead / 架构师', department='AI 研发部', email='zhang@xingchen.ai', agent_type='human', skills=['Python', '架构设计'])
add_contact(contacts, name='陈默', role='AI 应用开发工程师', department='AI 研发部', email='chenmo@xingchen.ai', agent_type='human', skills=['Python', 'JSON', 'Prompt'])
add_contact(contacts, name='ZhiShu-QA-Bot', role='自动化测试 Agent', department='质量保障部', email='qa-bot@xingchen.ai', agent_type='agent', skills=['pytest', 'CI'])
add_contact(contacts, name='ZhiShu-RAG-Bot', role='企业知识库检索 Agent', department='AI 研发部', email='rag-bot@xingchen.ai', agent_type='agent', skills=['RAG', 'VectorDB'])

print(format_contacts_table(list_contacts(contacts)))

姓名/标识            | 类型     | 部门           | 岗位/职责                | 技能标签
----------------------------------------------------------------------------------
张工               | human  | AI 研发部       | Tech Lead / 架构师      | Python, 架构设计
陈默               | human  | AI 研发部       | AI 应用开发工程师           | Python, JSON, Prompt
ZhiShu-QA-Bot    | agent  | 质量保障部        | 自动化测试 Agent          | pytest, CI
ZhiShu-RAG-Bot   | agent  | AI 研发部       | 企业知识库检索 Agent        | RAG, VectorDB


## 3. Read（检索与筛选）：模糊搜索与按类型/部门过滤
使用 `find_contact()` 按关键词检索，并使用 `list_contacts()` 组合筛选出所有 AI Agent 节点。

In [3]:
print('--- 检索关键词: JSON ---')
print(format_contacts_table(find_contact(contacts, 'JSON')))

print('\n--- 仅查看 AI 研发部的 Agent 节点 ---')
print(format_contacts_table(list_contacts(contacts, agent_type='agent', department='AI 研发部')))

--- 检索关键词: JSON ---
姓名/标识            | 类型     | 部门           | 岗位/职责                | 技能标签
----------------------------------------------------------------------------------
陈默               | human  | AI 研发部       | AI 应用开发工程师           | Python, JSON, Prompt

--- 仅查看 AI 研发部的 Agent 节点 ---
姓名/标识            | 类型     | 部门           | 岗位/职责                | 技能标签
----------------------------------------------------------------------------------
ZhiShu-RAG-Bot   | agent  | AI 研发部       | 企业知识库检索 Agent        | RAG, VectorDB


## 4. Update & Delete（更新与删除）：局部字段修改与安全移除
使用 `update_contact()` 更新陈默的岗位与技能列表，再使用 `delete_contact()` 移除指定 Agent 节点，并验证异常防御。

In [4]:
updated = update_contact(contacts, '陈默', role='高级 AI 应用开发工程师', skills=['Python', 'JSON', 'Agent'])
print('更新后:', updated)

removed = delete_contact(contacts, 'ZhiShu-QA-Bot')
print('已删除:', removed['name'])

try:
    add_contact(contacts, name='张工', role='重复人员')
except ValueError as err:
    print('防重名校验拦截成功:', err)

更新后: {'name': '陈默', 'role': '高级 AI 应用开发工程师', 'department': 'AI 研发部', 'email': 'chenmo@xingchen.ai', 'phone': '', 'agent_type': 'human', 'skills': ['Python', 'JSON', 'Agent']}
已删除: ZhiShu-QA-Bot
防重名校验拦截成功: 联系人已存在: 张工


## 5. JSON 持久化闭环：`save_contacts` 与 `load_contacts`
使用临时目录验证 `save_contacts()`（`ensure_ascii=False, indent=2`）与 `load_contacts()` 的完整读写闭环。

In [5]:
import tempfile

with tempfile.TemporaryDirectory() as tmp_dir:
    db_path = Path(tmp_dir) / 'zhishu_contacts.json'
    save_contacts(db_path, contacts)
    raw_json = db_path.read_text(encoding='utf-8')
    print('持久化 JSON 预览 (前 300 字符):')
    print(raw_json[:300])

    reloaded = load_contacts(db_path)
    assert reloaded == contacts
    print(f'\n重新加载成功，共 {len(reloaded)} 条联系人记录，数据 100% 一致！')

持久化 JSON 预览 (前 300 字符):
[
  {
    "name": "张工",
    "role": "Tech Lead / 架构师",
    "department": "AI 研发部",
    "email": "zhang@xingchen.ai",
    "phone": "",
    "agent_type": "human",
    "skills": [
      "Python",
      "架构设计"
    ]
  },
  {
    "name": "陈默",
    "role": "高级 AI 应用开发工程师",
    "department": "AI 研发部",
    

重新加载成功，共 3 条联系人记录，数据 100% 一致！
